## Model v4: Optuna Tuned XGBoost + CatBoost Stacking

In [1]:
# !pip install optuna -q

import os
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e5'
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e5/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e5/train.csv
/kaggle/input/competitions/playground-series-s6e5/test.csv


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import catboost as cb
import cupy as cp
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from scipy.stats import rankdata

optuna.logging.set_verbosity(optuna.logging.WARNING)

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f"Train: {train.shape}, Test: {test.shape}")
print(f"Columns: {list(train.columns)}")
print(f"Target distribution:\n{train['PitNextLap'].value_counts(normalize=True)}")

Train: (439140, 16), Test: (188165, 15)
Columns: ['id', 'Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'PitNextLap']
Target distribution:
PitNextLap
0.0    0.801018
1.0    0.198982
Name: proportion, dtype: float64


In [3]:
def add_features(df):
    """Enhanced feature engineering (v3 features + new features)."""
    df = df.copy()

    # --- v3 features ---
    df['TyreLife_x_Compound'] = df['TyreLife'].astype(str) + '_' + df['Compound']
    df['LapNumber_sq'] = df['LapNumber'] ** 2
    df['TyreLife_sq'] = df['TyreLife'] ** 2
    df['TyreLife_per_Progress'] = df['TyreLife'] / (df['RaceProgress'] + 1e-6)
    df['LapTime_vs_RaceMean'] = df.groupby('Race')['LapTime (s)'].transform(lambda x: x - x.mean())
    df['Stint_x_TyreLife'] = df['Stint'] * df['TyreLife']
    df['HighTyreLife'] = (df['TyreLife'] > 20).astype(int)
    df['PosGroup'] = pd.cut(df['Position'], bins=[0, 5, 10, 20], labels=['top', 'mid', 'back'])
    df['CumPitStops'] = df.groupby(['Race', 'Year', 'Driver'])['PitStop'].cumsum()
    df['LapsToGo'] = df.groupby(['Race', 'Year', 'Driver'])['LapNumber'].transform('max') - df['LapNumber']
    df['Delta_positive'] = (df['LapTime_Delta'] > 0).astype(int)
    df['Abs_LapTime_Delta'] = df['LapTime_Delta'].abs()
    df['RaceLapCount'] = df.groupby(['Race', 'Year'])['LapNumber'].transform('max')
    df['LapInRace'] = df['LapNumber'] / df['RaceLapCount']

    # --- v4 new features ---
    df['TyreLife_cubed'] = df['TyreLife'] ** 3

    df['LapTime_rolling3'] = (
        df.groupby(['Race', 'Year', 'Driver'])['LapTime (s)']
        .transform(lambda x: x.rolling(3, min_periods=1).mean())
    )

    df['LapTime_delta_diff'] = (
        df.groupby(['Race', 'Year', 'Driver'])['LapTime_Delta']
        .transform(lambda x: x.diff().fillna(0))
    )

    df['StintLength'] = (
        df.groupby(['Race', 'Year', 'Driver', 'Stint'])['TyreLife']
        .transform('max')
    )

    compound_avg_pit = (
        df[df['PitStop'] == 1]
        .groupby('Compound')['TyreLife']
        .median()
        .to_dict()
    )
    df['CompoundAvgPitLap'] = df['Compound'].map(compound_avg_pit).fillna(15)
    df['DistToPitWindow'] = (df['TyreLife'] - df['CompoundAvgPitLap']).abs()

    df['Position_pct'] = df['Position'] / df.groupby(['Race', 'Year'])['Position'].transform('max')

    df['TyreLife_bin'] = pd.cut(df['TyreLife'], bins=[0, 5, 10, 15, 20, 30, 100],
                                labels=['0-5', '6-10', '11-15', '16-20', '21-30', '30+'])

    return df


In [4]:
train['is_train'] = 1
test['is_train'] = 0
test['PitNextLap'] = np.nan
combined = pd.concat([train, test], ignore_index=True)
combined = add_features(combined)
train_fe = combined[combined['is_train'] == 1].drop(columns=['is_train'])
test_fe = combined[combined['is_train'] == 0].drop(columns=['is_train', 'PitNextLap'])

target = 'PitNextLap'
drop_cols = ['id', target]
features = [c for c in train_fe.columns if c not in drop_cols]
cat_features = ['Driver', 'Compound', 'Race', 'TyreLife_x_Compound', 'PosGroup', 'TyreLife_bin']

X = train_fe[features].copy()
y = train_fe[target].copy()
X_test = test_fe[features].copy()

for col in cat_features:
    if col in X.columns:
        X[col] = X[col].astype('category')
        X_test[col] = X_test[col].astype('category')

# XGBoost: category -> integer codes
X_xgb = X.copy()
X_test_xgb = X_test.copy()
for col in cat_features:
    if col in X_xgb.columns:
        X_xgb[col] = X_xgb[col].cat.codes
        X_test_xgb[col] = X_test_xgb[col].cat.codes

# CatBoost: category -> string
X_cb = X.copy()
X_test_cb = X_test.copy()
for col in cat_features:
    if col in X_cb.columns:
        X_cb[col] = X_cb[col].astype(str)
        X_test_cb[col] = X_test_cb[col].astype(str)

cat_idx = [features.index(c) for c in cat_features if c in features]

print(f"Features ({len(features)}): {features}")
print(f"Categorical ({len(cat_features)}): {cat_features}")
print(f"Train: {X.shape}, Test: {X_test.shape}")

Features (36): ['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'TyreLife_x_Compound', 'LapNumber_sq', 'TyreLife_sq', 'TyreLife_per_Progress', 'LapTime_vs_RaceMean', 'Stint_x_TyreLife', 'HighTyreLife', 'PosGroup', 'CumPitStops', 'LapsToGo', 'Delta_positive', 'Abs_LapTime_Delta', 'RaceLapCount', 'LapInRace', 'TyreLife_cubed', 'LapTime_rolling3', 'LapTime_delta_diff', 'StintLength', 'CompoundAvgPitLap', 'DistToPitWindow', 'Position_pct', 'TyreLife_bin']
Categorical (6): ['Driver', 'Compound', 'Race', 'TyreLife_x_Compound', 'PosGroup', 'TyreLife_bin']
Train: (439140, 36), Test: (188165, 36)


In [5]:
N_OPTUNA_TRIALS = 30
N_OPTUNA_FOLDS = 3

# Pre-convert XGBoost data to cupy (GPU) for all XGBoost operations
X_xgb_gpu = cp.asarray(X_xgb.values.astype(np.float32))
X_test_xgb_gpu = cp.asarray(X_test_xgb.values.astype(np.float32))

def objective_xgb(trial):
    params = {
        'n_estimators': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42, 'eval_metric': 'auc',
        'device': 'cuda', 'enable_categorical': True,
    }
    skf = StratifiedKFold(n_splits=N_OPTUNA_FOLDS, shuffle=True, random_state=42)
    scores = []
    for tr_idx, va_idx in skf.split(X, y):
        m = xgb.XGBClassifier(**params)
        m.fit(X_xgb_gpu[tr_idx], y.iloc[tr_idx],
              eval_set=[(X_xgb_gpu[va_idx], y.iloc[va_idx])], verbose=False)
        p = m.predict_proba(X_xgb_gpu[va_idx])[:, 1]
        p_np = cp.asnumpy(p) if hasattr(p, 'get') else p
        scores.append(roc_auc_score(y.iloc[va_idx], p_np))
    return np.mean(scores)

def objective_cb(trial):
    params = {
        'iterations': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'bootstrap_type': 'Bernoulli',
        'random_seed': 42, 'eval_metric': 'Logloss',
        'verbose': 0, 'early_stopping_rounds': 100,
        'task_type': 'GPU',
    }
    skf = StratifiedKFold(n_splits=N_OPTUNA_FOLDS, shuffle=True, random_state=42)
    scores = []
    for tr_idx, va_idx in skf.split(X, y):
        m = cb.CatBoostClassifier(**params)
        m.fit(X_cb.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=[(X_cb.iloc[va_idx], y.iloc[va_idx])],
              cat_features=cat_idx, verbose=0)
        p = m.predict_proba(X_cb.iloc[va_idx])[:, 1]
        scores.append(roc_auc_score(y.iloc[va_idx], p))
    return np.mean(scores)

print(f"Optuna: {N_OPTUNA_TRIALS} trials x {N_OPTUNA_FOLDS} folds per model (XGB + CB)")

Optuna: 30 trials x 3 folds per model (XGB + CB)


In [6]:
print("=== Tuning XGBoost ===")
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True) # type: ignore
print(f"Best XGB AUC: {study_xgb.best_value:.5f}")
print(f"Best XGB params: {study_xgb.best_params}")

print("\n=== Tuning CatBoost ===")
study_cb = optuna.create_study(direction='maximize')
study_cb.optimize(objective_cb, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True) # type: ignore
print(f"Best CB AUC: {study_cb.best_value:.5f}")
print(f"Best CB params: {study_cb.best_params}")

=== Tuning XGBoost ===


  0%|          | 0/30 [00:00<?, ?it/s]

Best XGB AUC: 0.94938
Best XGB params: {'learning_rate': 0.013298894760343469, 'max_depth': 9, 'subsample': 0.6801421090816778, 'colsample_bytree': 0.6007744459387401, 'reg_alpha': 0.03638640822372883, 'reg_lambda': 2.218906661803763}

=== Tuning CatBoost ===


  0%|          | 0/30 [00:00<?, ?it/s]

Best CB AUC: 0.94918
Best CB params: {'learning_rate': 0.026452995872180324, 'depth': 10, 'l2_leaf_reg': 6.635729205899109, 'subsample': 0.9672039193819852}


In [7]:
best_xgb_params = {
    **study_xgb.best_params,
    'n_estimators': 2000, 'random_state': 42, 'eval_metric': 'auc',
    'device': 'cuda', 'enable_categorical': True,
}
best_cb_params = {
    **study_cb.best_params,
    'iterations': 2000, 'bootstrap_type': 'Bernoulli',
    'random_seed': 42, 'eval_metric': 'Logloss', 'verbose': 0,
    'early_stopping_rounds': 100, 'task_type': 'GPU',
}

print("XGB params:", best_xgb_params)
print("CB params:", best_cb_params)

# 5-fold CV, collect OOF + test predictions
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

val_scores_xgb, val_scores_cb = [], []
test_preds_xgb, test_preds_cb = [], []
oof_xgb = np.zeros(len(X))
oof_cb = np.zeros(len(X))

# Pre-convert XGBoost data to cupy (GPU) to avoid CPU->GPU warning
X_xgb_gpu = cp.asarray(X_xgb.values.astype(np.float32))
X_test_xgb_gpu = cp.asarray(X_test_xgb.values.astype(np.float32))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

    # --- XGBoost (GPU data) ---
    m_xgb = xgb.XGBClassifier(**best_xgb_params)
    m_xgb.fit(X_xgb_gpu[train_idx], y_tr, eval_set=[(X_xgb_gpu[val_idx], y_va)], verbose=False)
    p_xgb = m_xgb.predict_proba(X_xgb_gpu[val_idx])[:, 1]
    oof_xgb[val_idx] = cp.asnumpy(p_xgb) if hasattr(p_xgb, 'get') else p_xgb
    val_scores_xgb.append(roc_auc_score(y_va, oof_xgb[val_idx]))
    t_xgb = m_xgb.predict_proba(X_test_xgb_gpu)[:, 1]
    test_preds_xgb.append(cp.asnumpy(t_xgb) if hasattr(t_xgb, 'get') else t_xgb)

    # --- CatBoost ---
    X_tr_c, X_va_c = X_cb.iloc[train_idx], X_cb.iloc[val_idx]
    m_cb = cb.CatBoostClassifier(**best_cb_params)
    m_cb.fit(X_tr_c, y_tr, eval_set=[(X_va_c, y_va)], cat_features=cat_idx, verbose=0)
    p_cb = m_cb.predict_proba(X_va_c)[:, 1]
    oof_cb[val_idx] = p_cb
    val_scores_cb.append(roc_auc_score(y_va, p_cb))
    test_preds_cb.append(m_cb.predict_proba(X_test_cb)[:, 1])

    print(f'Fold {fold+1}: XGB={val_scores_xgb[-1]:.5f}, CB={val_scores_cb[-1]:.5f}')

mean_xgb = np.mean(val_scores_xgb)
mean_cb = np.mean(val_scores_cb)
print(f'\nMean: XGB={mean_xgb:.5f}, CB={mean_cb:.5f}')

XGB params: {'learning_rate': 0.013298894760343469, 'max_depth': 9, 'subsample': 0.6801421090816778, 'colsample_bytree': 0.6007744459387401, 'reg_alpha': 0.03638640822372883, 'reg_lambda': 2.218906661803763, 'n_estimators': 2000, 'random_state': 42, 'eval_metric': 'auc', 'device': 'cuda', 'enable_categorical': True}
CB params: {'learning_rate': 0.026452995872180324, 'depth': 10, 'l2_leaf_reg': 6.635729205899109, 'subsample': 0.9672039193819852, 'iterations': 2000, 'bootstrap_type': 'Bernoulli', 'random_seed': 42, 'eval_metric': 'Logloss', 'verbose': 0, 'early_stopping_rounds': 100, 'task_type': 'GPU'}
Fold 1: XGB=0.95078, CB=0.95065
Fold 2: XGB=0.94916, CB=0.94889
Fold 3: XGB=0.94976, CB=0.94966
Fold 4: XGB=0.94946, CB=0.94891
Fold 5: XGB=0.95026, CB=0.94978

Mean: XGB=0.94989, CB=0.94958


In [8]:
# --- Method 1: Weighted average ---
total = mean_xgb + mean_cb
w_xgb = mean_xgb / total
w_cb = mean_cb / total
test_wavg = w_xgb * np.mean(test_preds_xgb, axis=0) + w_cb * np.mean(test_preds_cb, axis=0)
oof_wavg = w_xgb * oof_xgb + w_cb * oof_cb
print(f"Weighted avg: XGB={w_xgb:.4f}, CB={w_cb:.4f} -> OOF AUC={roc_auc_score(y, oof_wavg):.5f}")

# --- Method 2: Stacking (Logistic Regression) ---
meta_X = np.column_stack([oof_xgb, oof_cb])
meta_X_test = np.column_stack([np.mean(test_preds_xgb, axis=0), np.mean(test_preds_cb, axis=0)])

meta_lr = LogisticRegression(C=1.0, random_state=42)
meta_lr.fit(meta_X, y)
test_stack = meta_lr.predict_proba(meta_X_test)[:, 1]
oof_stack = meta_lr.predict_proba(meta_X)[:, 1]
print(f"Stacking LR:  coefs={meta_lr.coef_[0]}, intercept={meta_lr.intercept_[0]:.4f} -> OOF AUC={roc_auc_score(y, oof_stack):.5f}")

# --- Method 3: Rank average ---
rank_xgb = rankdata(np.mean(test_preds_xgb, axis=0)) / len(test_preds_xgb[0])
rank_cb = rankdata(np.mean(test_preds_cb, axis=0)) / len(test_preds_cb[0])
test_rankavg = w_xgb * rank_xgb + w_cb * rank_cb
oof_rankavg = w_xgb * rankdata(oof_xgb) / len(oof_xgb) + w_cb * rankdata(oof_cb) / len(oof_cb)
print(f"Rank avg:     -> OOF AUC={roc_auc_score(y, oof_rankavg):.5f}")

# Pick best
best_method = max(
    [('wavg', oof_wavg, test_wavg), ('stack', oof_stack, test_stack), ('rank', oof_rankavg, test_rankavg)],
    key=lambda x: roc_auc_score(y, x[1])
)
print(f"\nBest method: {best_method[0]} (OOF AUC={roc_auc_score(y, best_method[1]):.5f})")

Weighted avg: XGB=0.5001, CB=0.4999 -> OOF AUC=0.95041
Stacking LR:  coefs=[3.4309152  3.41229952], intercept=-3.6182 -> OOF AUC=0.95041
Rank avg:     -> OOF AUC=0.95038

Best method: stack (OOF AUC=0.95041)


In [9]:
test_final = best_method[2]

submission = pd.DataFrame({'id': test['id'], 'PitNextLap': test_final})
submission.to_csv('submission.csv', index=False)
print(f"Submission saved: {submission.shape}")
print(f"PitNextLap stats: min={test_final.min():.4f}, max={test_final.max():.4f}, mean={test_final.mean():.4f}")
submission.head()

Submission saved: (188165, 2)
PitNextLap stats: min=0.0261, max=0.9598, mean=0.1982


,id,PitNextLap
0,439140,0.026851
1,439141,0.026874
2,439142,0.026729
3,439143,0.078794
4,439144,0.915124
